### Setup

In [ ]:
# imports
import  nekt
import  os                           
from    dotenv         import load_dotenv      
from    typing         import Optional 
from    pyspark.sql    import DataFrame
from    pyspark.sql    import functions    as F
from    pyspark.shell  import spark

# get Nekt data access token        
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")



# auxiliary functions
def extract_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )



# bronze
## clickup
df_bronze_clickup_users        = extract_nekt_table("users"        , "Bronze")
df_bronze_clickup_spaces       = extract_nekt_table("spaces"       , "Bronze")
df_bronze_clickup_time_entries = extract_nekt_table("time_entries" , "Bronze")

## conta azul
df_bronze_contaazul_accounts_payable    = extract_nekt_table("despesas"             , "Bronze")
df_bronze_contaazul_accounts_receivable = extract_nekt_table("receitas"             , "Bronze")
df_bronze_contaazul_categories          = extract_nekt_table("categorias"           , "Bronze")
# df_bronze_contaazul_costs_centers       = extract_nekt_table("centros_de_custos"    , "Bronze")
# df_bronze_contaazul_financial_accounts  = extract_nekt_table("contas_financeiras"   , "Bronze")
df_bronze_contaazul_installments        = extract_nekt_table("parcelas"             , "Bronze")
# df_bronze_contaazul_people_details      = extract_nekt_table("pessoas_detalhes"     , "Bronze")
df_bronze_contaazul_people_list         = extract_nekt_table("pessoas_lista"        , "Bronze")
# df_bronze_contaazul_products            = extract_nekt_table("produtos"             , "Bronze")
# df_bronze_contaazul_services            = extract_nekt_table("servico"              , "Bronze")
df_bronze_contaazul_sales_details       = extract_nekt_table("vendas_detalhes"      , "Bronze")
df_bronze_contaazul_sales_items         = extract_nekt_table("vendas_lista"         , "Bronze")
df_bronze_contaazul_sales_list          = extract_nekt_table("vendas_itens"         , "Bronze")
df_bronze_contaazul_dre_categories      = extract_nekt_table("categorias_dre"       , "Bronze")



# silver
## clickup
### users
df_silver_clickup_users = df_bronze_clickup_users.select(
      F.col("id").cast("integer")     .alias("user_id")
    , F.col("username")               .alias("user_name")
).filter(
      F.col("user_name").isNotNull()
).dropDuplicates(
    ["user_id"]
)

### spaces
df_silver_clickup_spaces = df_bronze_clickup_spaces.select(
      F.col("id").cast("long")        .alias("space_id")
    , F.col("name")                   .alias("space_name")
).filter(
      F.col("space_name").isNotNull()
).dropDuplicates(
    ["space_id"]
)

### time entries
df_silver_clickup_time_entries = df_bronze_clickup_time_entries.select(
      F.col("id").cast("long")                        .alias("interval_id")
    , F.col("wid").cast("integer")                    .alias("team_id")
    , F.col("user.id").cast("integer")                .alias("user_id")
    , F.col("user.username")                          .alias("user_name")
    , F.col("task_location.space_id").cast("long")    .alias("space_id")
    , F.col("task.id").alias("task_id")
    , F.col("start").cast("long")                     .alias("interval_date_start_ms")
    , F.to_timestamp(F.col("start") / 1000)           .alias("interval_date_start_iso")
    , F.col("end").cast("long")                       .alias("interval_date_end_ms")
    , F.to_timestamp(F.col("end") / 1000)             .alias("interval_date_end_iso")
    , F.col("at").cast("long")                        .alias("interval_date_added_ms")
    , F.to_timestamp(F.col("at") / 1000)              .alias("interval_date_added_iso")
).filter(
      F.col("interval_id").isNotNull()
    & F.col("user_id").isNotNull()
    & F.col("interval_date_start_ms").isNotNull()
    & F.col("interval_date_end_ms").isNotNull()
    & (F.col("interval_date_end_ms") > F.col("interval_date_start_ms"))  # validating business rule
).dropDuplicates(
    ["interval_id"]
)



## conta azul 
### accounts payable (expenses)
df_silver_contaazul_accounts_payable = df_bronze_contaazul_accounts_payable.withColumn(
    "categoria", F.explode("categorias")
).select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    , F.col("categoria.id")     .alias("categoria_principal_id")
    , F.col("categoria.nome")   .alias("categoria_principal_nome")
    , F.col("fornecedor.id")    .alias("fornecedor_id")
    , F.col("fornecedor.nome")  .alias("fornecedor_nome")
    , F.current_timestamp()     .alias("_loaded_at")
).filter(
      F.col("id").isNotNull()
    & F.col("data_criacao").isNotNull()
).dropDuplicates(
    ["id"]
)

### accounts receivable (revenues)
df_silver_contaazul_accounts_receivable = df_bronze_contaazul_accounts_receivable.withColumn(
    "categorias", F.explode("categorias")
).select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    , F.col("categorias.id")    .alias("categoria_principal_id")
    , F.col("categorias.nome")  .alias("categoria_principal_nome")
    , F.col("cliente.id")       .alias("cliente_id")
    , F.col("cliente.nome")     .alias("cliente_nome")
    , F.current_timestamp()     .alias("_loaded_at")
).filter(
      F.col("id").isNotNull()
    & F.col("data_criacao").isNotNull()
).dropDuplicates(
    ["id"]
)

### categories         
df_silver_contaazul_categories = df_bronze_contaazul_categories.select(
      F.col("id")
    , F.col("nome")
    , F.col("versao")
    , F.col("categoria_pai")
    , F.col("tipo")
    , F.col("entrada_dre")
    , F.col("considera_custo_dre")
    , F.current_timestamp().alias("_loaded_at")
).dropDuplicates(
    ["id"]
)

### parent_categories
parent_categories_list = [
      ('34d160af-1685-4482-a6fa-56e8c146f166', 'Custo Variável'                          )
    , ('f03625a7-0d22-45ea-baa3-1c95eb98c321', 'Despesas Administrativas e Comerciais'   )
    , ('7b69a4dc-1541-4782-9d12-71a3d60d0abb', 'Despesas Financeiras'                    )
    , ('026f5cd8-249a-43cb-b9a5-5c450de98c49', 'Despesas Marketing'                      )
    , ('06547c87-a867-4502-ade3-58924f28c026', 'Distribuição de Lucros'                  )
    , ('4c30406e-d500-42c9-86d3-76bc0b51d61e', 'Impostos'                                )
    , ('77b170e7-5ef2-4846-b5d8-1a8c91552719', 'Outras Despesas'                         )
    , ('32a4351d-5ad4-4906-85d0-4db31c8e2017', 'Custo Operacional'                       )
    , ('652879aa-81fb-41fb-843a-d95f884ca0f6', 'Custos Variáveis'                        )
    , ('2dce7ace-64d0-455b-acad-8591c747d04d', 'Departamento Pessoal'                    )
    , ('68b7acb4-cc44-4e7e-b2bf-8d23ce03cefc', 'Receitas'                                )
    , ('95a964c2-f31a-4347-926a-d3f67052c586', 'Receitas de Vendas'                      )
    , ('b6e66e3a-6673-4ced-8dbd-6990154007d4', 'Receitas Financeiras'                    )
    , ('aad461bc-d8a2-41e8-aca2-5f2b1753d8fb', 'Operacional'                             )    
    , ('2e3003c8-ae36-4f30-8f98-8a878c756d81', 'Despesas Imóvel'                         )
    , ('624aa9a0-2727-4dbc-9921-f811e64ae69d', 'Investimentos'                           )  
    , ('10cef24a-2a13-46cd-8103-8cf9cccf2db5', 'Custo Fixo'                              )
    , ('e74f1435-43fa-4ba2-8004-c295cd55cf55', 'Custo Operacional'                       )
    , ('816d543c-5a57-48b3-beea-71efa64e21ca', 'Assessorias e Associações'               )
]

df_parent_categories = spark.createDataFrame(
    parent_categories_list, ["categoria_pai_id", "categoria_pai_nome"]
)

### customers ???

### combined accounts
df_silver_contaazul_combined_accounts = (
    df_silver_contaazul_accounts_payable.withColumn("tipo", F.lit("D"))
    .unionByName(
        df_silver_contaazul_accounts_receivable.withColumn("tipo", F.lit("R")),
        allowMissingColumns=True
    )
).join(
    # first join — get categoria_pai_id
    df_silver_contaazul_categories.select(
          F.col("id").alias("categoria_principal_id")
        , F.col("categoria_pai").alias("categoria_pai_id")
    ),
    on="categoria_principal_id",
    how="left"
).join(
    # second join — get categoria_pai_nome using categoria_pai_id
    df_parent_categories,
    on=df_parent_categories["categoria_pai_id"] == df_silver_contaazul_categories["categoria_pai"],
    how="left"
).show()

### DRE financial categoires
        
### installments
df_silver_contaazul_installments = df_bronze_contaazul_installments.select(
      F.col("id")                       .alias("parcela_id")
    , F.col("status")                   .alias("parcela_status")
    , F.col("evento.condicao_pagamento").alias("condicao_pagamento")
    , F.col("referencia")
    , F.col("evento.agendado")          .alias("agendado")
    , F.col("evento.tipo")              .alias("tipo_evento")
    # , F.col("")                         .alias("rateio")
    # , F.col("")                         .alias("conciliado")
    # , F.col("")                         .alias("valor_pago")
    # , F.col("")                         .alias("data_vencimento")
    # , F.col("")                         .alias("data_pagamento_previsto")
    # , F.col("")                         .alias("descricao")
    # , F.col("")                         .alias("id_conta_financeira")
    # , F.col("")                         .alias("metodo_pagamento")
    # , F.col("")                         .alias("parent_evento_id")
    # , F.col("")                         .alias("rateio_id_categoria")
    # , F.col("")                         .alias("rateio_nome_categoria")
    # , F.col("")                         .alias("rateio_valor")
    # , F.col("")                         .alias("rateio_centro_custo_id")
    # , F.col("")                         .alias("rateio_centro_custo_valor")
    , F.current_timestamp()             .alias("parcela_loaded_at")  
    , F.current_timestamp()             .alias("_loaded_at")    
).dropDuplicates(
    ["parcela_id"]
)  

### settled installments
# df_silver_contaazul_settled_installments = df_bronze_contaazul_installments.withColumn(
#     "baixas", F.explode("baixas")
# ).select(
#       F.col("parcela_id")
#     , F.col("baixa.id")                             .alias("baixa_id")
#     , F.col("baixa.versao")                         .alias("baixa_versao")
#     , F.col("baixa.data_pagamento")                 .alias("baixa_data_pagamento") 
#     , F.col("baixa.id_reconciliacao")               .alias("baixa_id_reconciliacao")
#     , F.col("baixa.id_solicitacao_cobranca")        .alias("baixa_solicitacao_cobranca")
#     , F.col("baixa.observacao")                     .alias("baixa_observacao")
#     , F.col("baixa.metodo_pagamento")               .alias("baixa_metodo_pagamento")
#     , F.col("baixa.origem")                         .alias("baixa_origem")
#     , F.col("baixa.id_recibo_digital")              .alias("baixa_id_recibo_digital")
#     , F.col("baixa.tipo_evento_financeiro")         .alias("baixa_tipo_evento_financeiro")
#     , F.col("baixa.nsu")                            .alias("baixa_nsu")
#     , F.col("baixa.id_referencia")                  .alias("baixa_id_referencia")
#     , F.col("baixa.atualizado_em")                  .alias("baixa_atualizado_em")
#     , F.col("baixa.valor_composicao.desconto")      .alias("baixa_desconto")
#     , F.col("baixa.valor_composicao.juros")         .alias("baixa_juros")
#     , F.col("baixa.valor_composicao.multa")         .alias("baixa_multa")
#     , F.col("baixa.valor_composicao.taxa")          .alias("baixa_taxa")
#     , F.col("baixa.valor_composicao.valor_bruto")   .alias("baixa_valor_bruto")
#     , F.col("baixa.valor_composicao.valor_liquido") .alias("baixa_valor_liquido")
#     , F.current_timestamp()                         .alias("loaded_at")     
# ).filter(
#       F.col("baixa.id").isNotNull()
# ).dropDuplicates(
#     ["parcela_id", "baixa_id"]
# )
      
### people_list
# df_silver_contaazul_people_list = df_bronze_contaazul_people_list.select(
# )         
# df_bronze_contaazul_products          
# df_bronze_contaazul_services          
# df_bronze_contaazul_sales_details       
# df_bronze_contaazul_sales_items         
# df_bronze_contaazul_sales_list



## saving tables
### clickup
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

### conta azul
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

params: {'include_layer_database_name': 'true'}
Table details: {'id': '7adcd28a-c0e3-4a6a-aea2-1eeef140cee7', 'name': 'users', 'slug': 'users', 'database_id': 'users', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.users', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'deleted': False, 'created_at': '2026-02-26T11:33:03.805939-03:00', 'updated_at': '2026-03-28T12:37:14.407918-03:00'}


Py4JJavaError: An error occurred while calling o65.load.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: bigquery. Please find packages at `https://spark.apache.org/third-party-projects.html`.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:725)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:647)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:697)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:208)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:172)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at 

### Saving